# Лабораторная работа №1

## Задачи компьютерного зрения, метрики, способы решения

### Цель

Научиться корректно ставить задачи классификации, детекции и сегментации на одном материале, подбирать соответствующие им метрики и сравнивать классический и нейросетевой подходы в одинаковых условиях. Результат работы — не максимальная accuracy, а обоснованное сравнение двух подходов по качеству, трудоёмкости и устойчивости к деградации входных данных.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

| Библиотека | Роль в работе |
|---|---|
| `numpy`, `opencv-python` | обработка изображений, модели искажений |
| `scikit-image` | признаки (HOG), тестовые изображения |
| `scikit-learn` | kNN, SVM, разбиение выборки, метрики классификации |
| `matplotlib`, `pandas` | визуализация, журнал экспериментов |
| `torch` / `torchvision` (**опционально**) | предобученная CNN без дообучения |

**Данные.** По заданию это небольшой размеченный набор: подмножество CIFAR-10 либо собственный набор из 2–3 классов. Загрузчик ниже устроен с явным резервом:

1. локальные батчи CIFAR-10 (`data/cifar-10-batches-py`) — читаются напрямую, без интернета и без `torch`;
2. `torchvision.datasets.CIFAR10(..., download=False)`, если данные уже скачаны в кэш;
3. **резерв:** синтетический набор из трёх классов геометрических фигур, генерируемый кодом.

Резервный набор позволяет пройти работу без интернета, но он проще реального: у него нет фона реальных сцен, вариаций освещения и ракурса. Если вы работаете на резерве, обязательно укажите это в отчёте как ограничение выводов. Как подключить реальные данные — в ячейке-инструкции раздела 4 и в [реестре датасетов блока](../../resources/datasets/README.md).

Отсутствие `torch` не ломает ноутбук: нейросетевая ветвь оборачивается в `try/except`, и работа выполняется на классическом пайплайне с явной пометкой о недоступности CNN.

In [ ]:
# Служебная ячейка: импорты, версии, seed.
import os
import time
import pickle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import skimage
from skimage.feature import hog

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, recall_score)

SEED = 42
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

AUTHOR = {"fio": "", "group": "", "work": "ЛР1"}   # TODO: заполните

DATA_ROOT = Path("data")
IMG_SIZE = 64          # все изображения приводятся к 64x64 для единообразия пайплайна

VERSIONS = {
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "scikit-image": skimage.__version__,
    "scikit-learn": sklearn.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
}
VERSIONS

## 2. Краткая теоретическая справка

### 2.1. Три задачи на одном материале

| Задача | Формат выхода | Основные метрики |
|---|---|---|
| Классификация | метка класса (или вектор вероятностей) на изображение | accuracy, macro F1, per-class recall, confusion matrix |
| Детекция | список $(class, x, y, w, h, score)$ | mAP по порогам IoU, precision/recall |
| Сегментация | маска $H \times W$ с меткой класса в каждом пикселе | IoU, Dice, boundary F-score |

Одно и то же изображение допускает все три постановки; различаются требуемая разметка, формат выхода и метрика. Применение accuracy к детекции — не «упрощение», а ошибка: у детекции нет фиксированного числа объектов, и понятие «правильный ответ на изображение» не определено.

### 2.2. Метрики классификации

$$ \mathrm{accuracy} = \frac{1}{N}\sum_{i=1}^{N} [\hat{y}_i = y_i], \qquad
F_1^{(c)} = \frac{2 P_c R_c}{P_c + R_c}, \qquad
\text{macro-}F_1 = \frac{1}{C}\sum_{c=1}^{C} F_1^{(c)} $$

Accuracy при несбалансированных классах вводит в заблуждение: при долях классов 90/10 тривиальный классификатор даёт 0.9. Macro-F1 усредняет по классам, а не по объектам, поэтому редкий класс имеет тот же вес, что и частый.

### 2.3. IoU, mAP, Dice

Для двух прямоугольников или двух масок:

$$ \mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}, \qquad
\mathrm{Dice}(A, B) = \frac{2|A \cap B|}{|A| + |B|} $$

Связь: $\mathrm{Dice} = \dfrac{2\,\mathrm{IoU}}{1 + \mathrm{IoU}}$. Dice систематически выше IoU при одном и том же перекрытии, поэтому сравнивать значения этих метрик между собой нельзя.

Average Precision — площадь под кривой precision-recall при заданном пороге IoU: обнаружения сортируются по убыванию уверенности, каждое объявляется TP (если IoU с ещё не занятым эталоном выше порога) или FP, затем

$$ \mathrm{AP} = \sum_k \bigl(R_k - R_{k-1}\bigr) P_k $$

mAP — усреднение AP по классам (и, в COCO-варианте, по порогам IoU от 0.50 до 0.95 с шагом 0.05).

### 2.4. Классический и нейросетевой подходы

Классический конвейер: **признаки, спроектированные вручную** (HOG, цветовые гистограммы, LBP) + классификатор (kNN, SVM). Признаки фиксированы и интерпретируемы, обучение занимает секунды, требуется мало данных, но качество ограничено выразительностью признаков.

Нейросетевой подход в этой работе — предобученная CNN **без дообучения**: сеть используется либо как классификатор своих 1000 классов ImageNet, либо как экстрактор признаков (выход предпоследнего слоя) с обучением поверх него простого классификатора. Второй вариант корректнее для набора с произвольными классами и обычно называется linear probing.

Сравнение подходов имеет смысл только при совпадении условий: **одно и то же разбиение, один и тот же test, одинаковая предобработка входа**. Сравнение классики на одном подмножестве с CNN на другом — типичная ошибка, обесценивающая работу целиком.

### 2.5. Трудоёмкость

Помимо качества сравниваются: время подготовки признаков и обучения, время инференса на объект, объём модели, число требуемых размеченных примеров, сложность внедрения (зависимости, необходимость GPU). Вывод «CNN лучше» без учёта этих факторов неполон.

## 3. Задачи

Формулировка по [методическим указаниям блока](README.md).

На небольшом размеченном наборе (например, подмножество CIFAR-10 или собственный набор из 2–3 классов):

1. Сформулируйте задачи классификации, детекции и сегментации на одном материале и подберите метрики каждой задачи (accuracy/F1, mAP/IoU, Dice).
2. Решите задачу классификации двумя способами — классическим (признаки + kNN/SVM) и готовой предобученной CNN (без дообучения).
3. Выполните предобработку зашумлённых изображений и покажите её влияние на качество.

**Ожидаемый результат:** сравнение классического и нейросетевого подходов по качеству и трудоёмкости; корректно посчитанные метрики.

**Что будет проверяться** ([рубрика](../teachers-assessment/README.md)): метрики соответствуют задачам (не accuracy для детекции); классический и нейросетевой подходы сравнены **на одном и том же разбиении**; влияние предобработки показано количественно.

## 4. Данные и фиксация разбиения

Ячейка ниже пытается загрузить CIFAR-10 и при неудаче переключается на синтетический набор. Источник данных печатается явно и записывается в журнал — при сдаче он должен быть указан в отчёте.

**Как подключить реальные данные.**

- CIFAR-10 без `torch`: скачайте архив `cifar-10-python.tar.gz` с [cs.toronto.edu/~kriz/cifar.html](https://www.cs.toronto.edu/~kriz/cifar.html), распакуйте в `data/` так, чтобы получился каталог `data/cifar-10-batches-py` с файлами `data_batch_1..5`, `test_batch`, `batches.meta`.
- С `torchvision`: `torchvision.datasets.CIFAR10(root="data", download=True)` один раз при наличии сети; далее загрузчик найдёт данные локально.
- Собственный набор из 2–3 классов: разложите изображения по каталогам `data/custom/<имя_класса>/*.png|jpg` и включите ветку загрузки в функции `load_dataset` (в коде отмечено местом `TODO`). Для собственного набора обязательно укажите в отчёте источник, число снимков на класс и условия съёмки.

In [ ]:
# Служебная ячейка: загрузка данных с резервом. Изменять требуется только в отмеченном месте.

CIFAR_CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck")


def _load_cifar_local(root: Path, wanted: tuple, n_per_class: int) -> tuple:
    '''Прочитать батчи CIFAR-10 напрямую из pickle. Возвращает (X, y, names) или None.'''
    batch_dir = root / "cifar-10-batches-py"
    if not batch_dir.exists():
        return None
    images, labels = [], []
    for name in ["data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4",
                 "data_batch_5", "test_batch"]:
        path = batch_dir / name
        if not path.exists():
            continue
        with open(path, "rb") as f:
            batch = pickle.load(f, encoding="bytes")
        raw = batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        images.append(raw)
        labels.append(np.array(batch[b"labels"]))
    if not images:
        return None
    images = np.concatenate(images)
    labels = np.concatenate(labels)
    wanted_idx = [CIFAR_CLASSES.index(c) for c in wanted]
    X, y = [], []
    for new_label, old_label in enumerate(wanted_idx):
        pos = np.flatnonzero(labels == old_label)[:n_per_class]
        X.append(images[pos])
        y.append(np.full(len(pos), new_label))
    return np.concatenate(X), np.concatenate(y), list(wanted)


def make_shape_dataset(n_per_class: int = 200, size: int = IMG_SIZE, seed: int = SEED) -> tuple:
    '''Резервный синтетический набор: круг / квадрат / треугольник на текстурном фоне.

    Выход: (X uint8 [N, size, size, 3], y int [N], names list[str],
            boxes float [N, 4] в формате (x, y, w, h), masks bool [N, size, size])
    Разметка боксов и масок точная — она нужна для постановки задач детекции и
    сегментации на том же материале (задание 1).
    '''
    rng = np.random.default_rng(seed)
    names = ["circle", "square", "triangle"]
    X, y, boxes, masks = [], [], [], []
    for label, _ in enumerate(names):
        for _ in range(n_per_class):
            bg = rng.integers(60, 160)
            img = np.full((size, size, 3), bg, dtype=np.uint8)
            img = np.clip(img.astype(np.float64) + rng.normal(0, 10, img.shape),
                          0, 255).astype(np.uint8)
            color = tuple(int(v) for v in rng.integers(160, 256, size=3))
            r = int(rng.integers(size // 6, size // 3))
            cx = int(rng.integers(r + 2, size - r - 2))
            cy = int(rng.integers(r + 2, size - r - 2))
            mask = np.zeros((size, size), dtype=np.uint8)
            if label == 0:
                cv2.circle(img, (cx, cy), r, color, -1)
                cv2.circle(mask, (cx, cy), r, 1, -1)
            elif label == 1:
                pt1, pt2 = (cx - r, cy - r), (cx + r, cy + r)
                cv2.rectangle(img, pt1, pt2, color, -1)
                cv2.rectangle(mask, pt1, pt2, 1, -1)
            else:
                pts = np.array([[cx, cy - r], [cx - r, cy + r], [cx + r, cy + r]], np.int32)
                cv2.fillPoly(img, [pts], color)
                cv2.fillPoly(mask, [pts], 1)
            ys, xs = np.nonzero(mask)
            boxes.append([float(xs.min()), float(ys.min()),
                          float(xs.max() - xs.min() + 1), float(ys.max() - ys.min() + 1)])
            masks.append(mask.astype(bool))
            X.append(img)
            y.append(label)
    return (np.array(X), np.array(y), names,
            np.array(boxes, dtype=np.float64), np.array(masks))


def load_dataset(wanted=("airplane", "cat", "ship"), n_per_class: int = 200) -> dict:
    '''Загрузить набор с резервом. Возвращает dict с ключами X, y, names, source, extras.'''
    # TODO (опционально): ветка загрузки собственного набора из data/custom/<класс>/
    local = _load_cifar_local(DATA_ROOT, wanted, n_per_class)
    source = "cifar10_local"
    if local is None:
        try:
            from torchvision.datasets import CIFAR10
            ds = CIFAR10(root=str(DATA_ROOT), train=True, download=False)
            images = np.array(ds.data)
            labels = np.array(ds.targets)
            wanted_idx = [CIFAR_CLASSES.index(c) for c in wanted]
            Xs, ys = [], []
            for new_label, old_label in enumerate(wanted_idx):
                pos = np.flatnonzero(labels == old_label)[:n_per_class]
                Xs.append(images[pos])
                ys.append(np.full(len(pos), new_label))
            local = (np.concatenate(Xs), np.concatenate(ys), list(wanted))
            source = "cifar10_torchvision"
        except Exception as exc:   # нет torchvision или нет данных в кэше
            print("CIFAR-10 недоступен:", type(exc).__name__, "-> используется резервный набор.")
            local = None
            source = "synthetic_shapes"

    if local is None:
        X, y, names, boxes, masks = make_shape_dataset(n_per_class=n_per_class)
        extras = {"boxes": boxes, "masks": masks}
    else:
        X, y, names = local
        extras = {}
    X = np.stack([cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
                  for img in X])
    return {"X": X, "y": np.asarray(y), "names": names, "source": source, "extras": extras}


DATASET = load_dataset()
print("Источник данных:", DATASET["source"], "| классы:", DATASET["names"],
      "| объектов:", len(DATASET["y"]), "| форма:", DATASET["X"].shape)
if DATASET["source"] == "synthetic_shapes":
    print("ВНИМАНИЕ: работа выполняется на резервном синтетическом наборе. "
          "Укажите это в отчёте как ограничение выводов.")

In [ ]:
# Служебная ячейка: фиксация разбиения (train / val / test). Изменять не требуется.
# Разбиение стратифицированное и фиксированное seed: оба подхода обязаны
# оцениваться на одном и том же test.

X_all, y_all = DATASET["X"], DATASET["y"]
idx_all = np.arange(len(y_all))

idx_train, idx_hold = train_test_split(idx_all, test_size=0.4, stratify=y_all, random_state=SEED)
idx_val, idx_test = train_test_split(idx_hold, test_size=0.5, stratify=y_all[idx_hold],
                                     random_state=SEED)

SPLIT = {"train": idx_train, "val": idx_val, "test": idx_test}
print({k: len(v) for k, v in SPLIT.items()})
print(pd.DataFrame({part: pd.Series(y_all[idx]).value_counts().sort_index()
                    for part, idx in SPLIT.items()}))

fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for ax, i in zip(axes.ravel(), RNG.choice(idx_train, size=12, replace=False)):
    ax.imshow(X_all[i])
    ax.set_title(DATASET["names"][y_all[i]], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Служебная ячейка: журнал экспериментов и метрики. Изменять не требуется.
RUNS: list = []


def log_run(**fields) -> dict:
    row = {"seed": SEED, "data_source": DATASET["source"], **fields}
    if isinstance(row.get("params"), dict):
        row["params"] = ", ".join(f"{k}={v}" for k, v in row["params"].items())
    RUNS.append(row)
    return row


def runs_table(stage: str = None) -> pd.DataFrame:
    df = pd.DataFrame(RUNS)
    if stage is not None and not df.empty:
        df = df[df["stage"] == stage]
    return df.reset_index(drop=True)


def classification_metrics(y_true, y_pred) -> dict:
    '''accuracy, macro F1 и минимальный per-class recall.'''
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 4),
        "min_class_recall": round(float(np.min(recall_score(y_true, y_pred, average=None))), 4),
    }


def iou_bbox(box_a, box_b) -> float:
    '''IoU двух боксов в формате (x, y, w, h).'''
    ax, ay, aw, ah = box_a
    bx, by, bw, bh = box_b
    inter_w = max(0.0, min(ax + aw, bx + bw) - max(ax, bx))
    inter_h = max(0.0, min(ay + ah, by + bh) - max(ay, by))
    inter = inter_w * inter_h
    union = aw * ah + bw * bh - inter
    return float(inter / union) if union > 0 else 0.0


def dice_mask(mask_true: np.ndarray, mask_pred: np.ndarray) -> float:
    '''Dice для бинарных масок.'''
    a = mask_true.astype(bool)
    b = mask_pred.astype(bool)
    denominator = a.sum() + b.sum()
    return 1.0 if denominator == 0 else float(2 * np.logical_and(a, b).sum() / denominator)


def iou_mask(mask_true: np.ndarray, mask_pred: np.ndarray) -> float:
    a = mask_true.astype(bool)
    b = mask_pred.astype(bool)
    union = np.logical_or(a, b).sum()
    return 1.0 if union == 0 else float(np.logical_and(a, b).sum() / union)


print("Журнал инициализирован.")

## 5. Задание 1. Три задачи на одном материале

Сначала — постановка. Для каждой из трёх задач опишите: что подаётся на вход, что является выходом, какая разметка требуется, какая метрика применяется и почему именно она. Заполните структуру `TASK_SPEC` ниже — она станет частью отчёта.

Затем — метрики. `iou_bbox`, `iou_mask` и `dice_mask` уже реализованы выше; недостающая часть — average precision, её нужно написать самостоятельно и проверить на игрушечном примере с известным ответом.

Если вы работаете на резервном наборе, эталонные боксы и маски уже есть (`DATASET["extras"]`). Для CIFAR-10 разметки детекции и сегментации нет: сформируйте её сами для небольшого подмножества (10–15 изображений) — например, грубой пороговой сегментацией с ручной проверкой — и честно опишите, как она получена и какова её точность.

In [ ]:
# TODO (задание 1.1): постановка трёх задач на одном материале.
# Заполните описания. Этот словарь войдёт в отчёт.

TASK_SPEC = {
    "classification": {
        "input": "",          # что подаётся на вход
        "output": "",         # формат выхода
        "annotation": "",     # какая разметка требуется
        "metrics": [],        # список метрик
        "why": "",            # почему эти метрики, что они не покажут
    },
    "detection": {
        "input": "", "output": "", "annotation": "", "metrics": [], "why": "",
    },
    "segmentation": {
        "input": "", "output": "", "annotation": "", "metrics": [], "why": "",
    },
}

pd.DataFrame(TASK_SPEC).T

In [ ]:
# TODO (задание 1.2): реализуйте average precision и продемонстрируйте метрики.

def average_precision(predictions, ground_truth, iou_threshold: float = 0.5) -> float:
    '''Average Precision для одного класса при заданном пороге IoU.

    Вход:
        predictions  : list[(box, score)], box = (x, y, w, h)
        ground_truth : list[box] — эталонные объекты того же изображения (или набора)
        iou_threshold: порог, при котором обнаружение считается верным
    Выход:
        float AP в [0, 1].
    Порядок действий:
        1. Отсортировать обнаружения по убыванию score.
        2. Идти по списку: обнаружение — TP, если IoU с ещё НЕ занятым эталоном
           выше порога (один эталон засчитывается не более одного раза), иначе FP.
        3. Накопить кумулятивные precision и recall.
        4. Проинтегрировать кривую precision-recall.
    Проверьте реализацию на примере с известным ответом: одно эталонное поле,
    два обнаружения (одно точное с высоким score, одно случайное) -> AP = 1.0
    при пороге 0.5; при обратном порядке score AP = 0.5.
    '''
    raise NotImplementedError


# TODO: продемонстрируйте все три группы метрик на реальных данных набора:
# - accuracy/macro F1 на предсказаниях любого классификатора (см. раздел 6);
# - IoU/mAP на 10-15 изображениях с боксами;
# - IoU/Dice на масках; покажите численно соотношение Dice = 2*IoU/(1+IoU).
# Если разметку детекции/сегментации вы строили сами, опишите процедуру и её погрешность.

## 6. Задание 2. Классический подход: признаки + классификатор

Рабочий пример ниже выполняет полный проход: HOG-признаки на обучающей части, kNN, оценка на **валидации**. Это baseline, относительно которого оцениваются все дальнейшие изменения.

Test в примере не используется намеренно: подбор признаков и гиперпараметров ведётся по валидации, и только финальные конфигурации оцениваются на test один раз (п. 2 [общих МУ](../../../docs/guidelines-students.md)). Повторный подбор по test делает оценку оптимистичной.

In [ ]:
# Рабочий пример (рельсы): HOG + kNN, оценка на валидации.

def to_gray(images: np.ndarray) -> np.ndarray:
    return np.stack([cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) for img in images])


def hog_features(images: np.ndarray) -> np.ndarray:
    '''HOG-дескрипторы набора изображений. Выход: [N, D] float64.'''
    gray = to_gray(images)
    return np.stack([hog(g, orientations=9, pixels_per_cell=(8, 8),
                         cells_per_block=(2, 2), feature_vector=True) for g in gray])


t0 = time.perf_counter()
F_train = hog_features(X_all[SPLIT["train"]])
F_val = hog_features(X_all[SPLIT["val"]])
feature_ms = (time.perf_counter() - t0) * 1e3

baseline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
baseline.fit(F_train, y_all[SPLIT["train"]])
pred_val = baseline.predict(F_val)

metrics = classification_metrics(y_all[SPLIT["val"]], pred_val)
log_run(stage="classification", approach="classic", method="hog+knn5", split="val",
        params={"orientations": 9, "ppc": 8, "k": 5}, feature_ms=round(feature_ms, 1), **metrics)

print("Размерность признаков:", F_train.shape[1], "| время извлечения, мс:", round(feature_ms))
print(classification_report(y_all[SPLIT["val"]], pred_val, target_names=DATASET["names"]))
runs_table("classification")

In [ ]:
# TODO (задание 2.1): расширьте классический подход.
#
# Требования:
# 1. Не менее двух наборов признаков (например, HOG и цветовые гистограммы в HSV,
#    либо их конкатенация). Реализуйте единый интерфейс extract_features.
# 2. Не менее двух классификаторов (kNN и SVM), подбор гиперпараметров по валидации:
#    k для kNN, C и gamma для SVC(kernel="rbf").
# 3. Все промежуточные результаты — в журнал через log_run(split="val", ...).
# 4. На test выносится только финальная конфигурация, один раз.

def extract_features(images: np.ndarray, kind: str = "hog", **params) -> np.ndarray:
    '''Извлечь признаки из набора изображений.

    Вход:
        images : np.ndarray uint8 [N, H, W, 3]
        kind   : "hog" | "color_hist" | "hog+color_hist"
        params : параметры дескрипторов
    Выход:
        np.ndarray float [N, D]; D одинаково для всех вызовов с одними params.
    Требование: функция не должна зависеть от разбиения — признаки для train, val
    и test извлекаются одним и тем же кодом с одними и теми же параметрами.
    Нормализацию (StandardScaler) настраивайте ТОЛЬКО на train (утечка через
    нормализацию по всей выборке — распространённая ошибка).
    '''
    raise NotImplementedError


# TODO: подбор по валидации и запись результатов

## 7. Задание 2. Предобученная CNN без дообучения

Ячейка ниже определяет доступность `torch`/`torchvision`. Если библиотеки или веса недоступны, работа продолжается: нейросетевая ветвь помечается как невыполненная, и вместо неё сравниваются два классических конвейера с разными признаками. В этом случае в отчёте нужно прямо указать, что сравнение «классика против CNN» не проведено, и не переносить выводы на нейросетевой подход.

Корректный способ применения предобученной сети к своим классам — **linear probing**: взять выход предпоследнего слоя как вектор признаков и обучить поверх него линейный классификатор на тех же `train`/`val`, что и классика. Использование готовых 1000 классов ImageNet напрямую допустимо только если ваши классы им соответствуют, и это нужно обосновать.

Веса требуют загрузки из сети при первом обращении. Если сети нет, `torchvision` бросит исключение — оно перехватывается.

In [ ]:
# Служебная ячейка: определение доступности нейросетевой ветви.
HAS_TORCH = False
TORCH_NOTE = ""
try:
    import torch
    import torchvision
    HAS_TORCH = True
    TORCH_NOTE = f"torch {torch.__version__}, torchvision {torchvision.__version__}"
except Exception as exc:
    TORCH_NOTE = f"torch недоступен ({type(exc).__name__}): нейросетевая ветвь пропускается"

print(TORCH_NOTE)
print("HAS_TORCH =", HAS_TORCH)
if not HAS_TORCH:
    print("Резервный план: вместо CNN сравните два классических конвейера с разными "
          "признаками и явно отметьте в отчёте, что сравнение с CNN не выполнено.")

In [ ]:
# TODO (задание 2.2): признаки предобученной CNN (linear probing).

def cnn_features(images: np.ndarray, model_name: str = "resnet18",
                 batch_size: int = 32) -> np.ndarray:
    '''Извлечь признаки предобученной сетью без дообучения.

    Вход:
        images : np.ndarray uint8 [N, H, W, 3] в порядке RGB
    Выход:
        np.ndarray float32 [N, D] — активации предпоследнего слоя.
    Порядок действий:
        1. Загрузить предобученную модель (torchvision.models), перевести в eval().
        2. Подготовить вход: изменение размера до требуемого моделью, приведение
           к float, нормализация средним и стандартным отклонением ImageNet,
           перестановка осей в [N, C, H, W].
        3. Отключить градиенты (torch.no_grad()).
        4. Снять активации предпоследнего слоя (например, заменив последний
           линейный слой на torch.nn.Identity()).
        5. Вернуть numpy-массив.
    Если HAS_TORCH == False, функция вызываться не должна: ветвь пропускается.
    '''
    raise NotImplementedError


# TODO: если HAS_TORCH — извлеките признаки для train/val/test ТЕХ ЖЕ индексов SPLIT,
# обучите линейный классификатор (LogisticRegression или LinearSVC) на train,
# подберите регуляризацию по val, запишите результаты в журнал с approach="cnn".
# Замерьте: время извлечения признаков на объект и время обучения классификатора.

In [ ]:
# TODO (задание 2.3): сравнение подходов на одном и том же test.
#
# Требования (проверяются по рубрике):
# 1. Оба подхода оцениваются на SPLIT["test"] — тех же индексах, одним вызовом.
# 2. Финальные конфигурации зафиксированы ДО обращения к test.
# 3. В таблицу сравнения входят не только метрики качества, но и трудоёмкость:
#    время подготовки признаков, время обучения, время инференса на объект,
#    размер модели, число зависимостей.
# 4. Приведена матрица ошибок для каждого подхода и разобраны классы,
#    на которых подходы расходятся сильнее всего.
#
# Каркас:
# for approach, features in [("classic", ...), ("cnn", ...)]:
#     ...
#     log_run(stage="classification", approach=approach, split="test",
#             method=..., params=..., **classification_metrics(y_test, pred))

# TODO: код сравнения

## 8. Задание 3. Предобработка зашумлённых изображений

Схема эксперимента: test-выборка искажается моделью деградации, затем оценивается качество (а) без предобработки и (б) после предобработки. Модели классификаторов при этом **не переобучаются** — иначе измеряется не эффект предобработки, а эффект дообучения на искажённых данных. Если вы хотите проверить и этот вариант, вынесите его в отдельную серию и обозначьте как отдельный фактор.

Три числа, которые должны быть в отчёте для каждой конфигурации: качество на чистом test (верхняя граница), качество на искажённом без предобработки (нижняя граница), качество на искажённом с предобработкой. Без первых двух третье интерпретировать невозможно.

Предупреждение: агрессивная предобработка (сильное сглаживание) снижает качество и на чистых данных. Проверьте это — типичный источник ошибочного вывода «фильтрация всегда полезна».

In [ ]:
# Служебная ячейка: модели деградации + рабочий пример (рельсы).

def degrade(images: np.ndarray, kind: str, level: float, seed: int = SEED) -> np.ndarray:
    '''Внести искажение в набор изображений.

    kind: "gaussian" (level = sigma), "salt_pepper" (level = доля пикселей),
          "blur" (level = sigma гауссова размытия), "lowlight" (level = коэффициент яркости)
    Выход: uint8 массив той же формы.
    '''
    rng = np.random.default_rng(seed)
    out = images.astype(np.float64).copy()
    if kind == "gaussian":
        out += rng.normal(0.0, level, out.shape)
    elif kind == "salt_pepper":
        mask = rng.random(out.shape[:3])
        out[mask < level / 2] = 255
        out[(mask >= level / 2) & (mask < level)] = 0
    elif kind == "blur":
        out = np.stack([cv2.GaussianBlur(img, (0, 0), level) for img in images]).astype(np.float64)
    elif kind == "lowlight":
        out = out * level
    else:
        raise ValueError(f"неизвестный вид искажения: {kind}")
    return np.clip(out, 0, 255).astype(np.uint8)


X_test_clean = X_all[SPLIT["test"]]
y_test = y_all[SPLIT["test"]]
X_test_noisy = degrade(X_test_clean, "salt_pepper", 0.05)

for tag, batch in [("clean", X_test_clean), ("salt_pepper_005", X_test_noisy)]:
    pred = baseline.predict(hog_features(batch))
    log_run(stage="robustness", approach="classic", method="hog+knn5", split="test",
            degradation=tag, preprocessing="none",
            params={"k": 5}, **classification_metrics(y_test, pred))

fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
for j in range(5):
    axes[0, j].imshow(X_test_clean[j]); axes[0, j].axis("off")
    axes[1, j].imshow(X_test_noisy[j]); axes[1, j].axis("off")
axes[0, 0].set_title("чистые", fontsize=9)
axes[1, 0].set_title("соль-перец 0.05", fontsize=9)
plt.tight_layout()
plt.show()
runs_table("robustness")

In [ ]:
# TODO (задание 3.1): предобработка и её количественное влияние.
#
# Обязательная серия:
#   искажения      : {"none", "gaussian" (2 уровня), "salt_pepper" (2 уровня), "blur", "lowlight"}
#   предобработка  : {"none", "median", "gaussian", "bilateral", "clahe"/выравнивание гистограммы}
#   подходы        : classic (обязательно) и cnn (если доступен)
#   выборка        : SPLIT["test"], модели НЕ переобучаются
# Метрики: accuracy, macro F1, min per-class recall; в журнал через
# log_run(stage="robustness", degradation=..., preprocessing=..., ...).

def preprocess(images: np.ndarray, method: str, **params) -> np.ndarray:
    '''Предобработать набор изображений перед подачей в классификатор.

    Вход:  uint8 [N, H, W, 3], method: "none" | "median" | "gaussian" | "bilateral" | "clahe"
    Выход: uint8 [N, H, W, 3] той же формы.
    Требование: одна и та же предобработка применяется ко всем изображениям набора;
    подбор параметров под отдельные изображения запрещён.
    '''
    raise NotImplementedError


# TODO: серия и таблица «искажение x предобработка -> метрики» для каждого подхода.
# Обязательно включите строку (чистые данные, предобработка) — она показывает,
# не вредит ли предобработка в отсутствие искажений.

## Отчёт

**Таблица 1.** Постановка трёх задач: задача → вход → выход → разметка → метрики → обоснование выбора метрик (`TASK_SPEC`).

**Таблица 2 — главная.** Сравнение подходов на одном test: подход → признаки/модель → accuracy, macro F1, min per-class recall → время признаков, время обучения, время инференса на объект → зависимости. Классический и нейросетевой подходы должны стоять в одной таблице на одних и тех же данных.

**Таблица 3.** Устойчивость: искажение × предобработка × подход → метрики. Обязательные опорные строки — чистый test и искажённый без предобработки.

Дополнительно: матрицы ошибок обоих подходов, примеры изображений, на которых подходы дают разные ответы.

Разделяйте наблюдение, интерпретацию и вывод (п. 2 [общих МУ](../../../docs/guidelines-students.md)). Если нейросетевая ветвь не выполнялась из-за отсутствия `torch`, это ограничение указывается в выводах явно, и утверждения о преимуществах CNN не делаются.

In [ ]:
# Сводные таблицы из журнала.
classification_runs = runs_table("classification")
if not classification_runs.empty:
    display(classification_runs)
    test_only = classification_runs[classification_runs["split"] == "test"]
    if not test_only.empty:
        display(test_only.groupby(["approach", "method"])[
            ["accuracy", "macro_f1", "min_class_recall"]].mean().round(4))
else:
    print("Журнал классификации пуст: выполните задания 2.1-2.3.")

robustness_runs = runs_table("robustness")
if not robustness_runs.empty:
    display(robustness_runs.pivot_table(index=["approach", "degradation"],
                                        columns="preprocessing",
                                        values="macro_f1", aggfunc="mean").round(4))

# TODO: постройте матрицы ошибок (confusion_matrix + imshow) для финальных
# конфигураций обоих подходов и график macro F1 по уровням искажения.

### Выводы

**Использованные данные:** источник (`DATASET["source"]`), число объектов, разбиение, seed.

**Наблюдения**

1.
2.
3.

**Интерпретация**

1.
2.

**Выводы и границы применимости**

1. Качество: какой подход и на сколько выигрывает на этом test при этом разбиении.
2. Трудоёмкость: время, зависимости, требования к данным.
3. Устойчивость: как меняется картина при искажениях и помогает ли предобработка.
4. Ограничения: размер выборки, единственное разбиение, отсутствие дообучения, резервный набор данных (если применимо).

**Анализ ошибок.** Не менее трёх изображений, на которых подходы расходятся, с разбором вероятной причины. Отдельно — класс с наименьшим recall и объяснение, чем он путается.

**Использование сторонних материалов и LLM.** Укажите источники данных, весов моделей и заимствованного кода (п. 5 общих МУ).

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока), относящиеся к этой работе:

8. Чем задачи классификации, детекции и сегментации отличаются по формату выхода и метрикам?
9. Что измеряют IoU и mAP?

Дополнительно к защите: почему accuracy непригодна для детекции? Чем отличаются Dice и IoU и почему их значения нельзя сравнивать напрямую? Почему предобученная сеть без дообучения может проигрывать HOG+SVM на конкретном наборе?

## Чек-лист перед сдачей

Полный список — в [общих МУ, п. 6](../../../docs/guidelines-students.md#6-чек-лист-перед-сдачей). Специфика ЛР1:

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`, в том числе при отсутствии `torch`.
- [ ] Указан источник данных (CIFAR-10 или резервный набор) и это учтено в выводах.
- [ ] Разбиение train/val/test зафиксировано; test использован один раз.
- [ ] Три задачи сформулированы, метрики каждой обоснованы; accuracy не применяется к детекции.
- [ ] Реализована и проверена average precision.
- [ ] Классический подход: не менее двух наборов признаков и двух классификаторов, подбор по валидации.
- [ ] Нейросетевой подход выполнен на **тех же** индексах разбиения (или его отсутствие явно оговорено).
- [ ] Есть сравнение по трудоёмкости, а не только по качеству.
- [ ] Влияние предобработки показано количественно, с опорными строками (чистый / искажённый без предобработки).
- [ ] Наблюдения отделены от интерпретаций, ограничения указаны.